In [6]:
import numpy as np
from itertools import combinations
import matplotlib.pyplot as plt

In [5]:
def generate_subsets(p):
    """
    Returns the list of all subsets.
    Order is important and must stay fixed during the algorithm.
    """
    subsets = []
    for k in range(p + 1):
        subsets.extend(combinations(range(p), k))
    return subsets


def mobius_inverse_matrix(subsets):
    """
    Construct M^{-1} using the closed-form formula:
        (M^{-1})_{T,U} = (-1)^{|U|-|T|} if T ⊆ U else 0
    """
    q = len(subsets)
    M_inv = np.zeros((q, q))

    for i, T in enumerate(subsets):
        set_T = set(T)

        for j, U in enumerate(subsets):
            if set_T.issubset(U):
                M_inv[i, j] = (-1) ** (len(U) - len(T))

    return M_inv

def pick_freeze_sample(sampler, f, U):
    """
    sampler : returns X ~ Q
    f       : function R^p -> R
    U       : frozen subset
    """

    X = sampler()
    X_prime = sampler()

    X_U = X.copy()
    X_U[list(U)] = X_prime[list(U)]

    Y = f(X)
    Y_U = f(X_U)

    return Y, Y_U

class PickFreezeMirror:

    def __init__( self, f, sampler, p, eta0=0.1, a=None):

        self.f = f
        self.sampler = sampler

        # --- subsets and Mobius matrix ---
        self.subsets = generate_subsets(p)
        self.q = len(self.subsets)

        self.M_inv = mobius_inverse_matrix(self.subsets)

        # -------------------------------------------------
        # Distribution "a" over subsets
        # -------------------------------------------------

        if a is None:
            self.a = np.ones(self.q) / self.q
        else:
            assert len(a) == self.q
            assert np.isclose(np.sum(a), 1)
            self.a = a

        # -------------------------------------------------
        # Algorithm initialization (paper)
        # m_0 = 0
        # S_0 uniform on simplex
        # -------------------------------------------------

        self.S = np.ones(self.q) / self.q
        self.m = 0.0
        self.n = 0

        self.eta0 = eta0

    # =====================================================
    # Gradient estimator from the paper
    # =====================================================

    def gradient(self, Y, YU, subset_index):

        """
        Implements:

        grad = (Y-m) M^{-1}_{U,:}
               * [ (Y-m) (M^{-1} S)_U - (Y^U - m) ]
        """

        delta = Y - self.m

        # row of Mobius inverse
        row = self.M_inv[subset_index]

        # compute v = M^{-1} S
        v = self.M_inv @ self.S

        scalar = delta * v[subset_index] - (YU - self.m)

        grad = delta * row * scalar

        return grad


    # =====================================================
    # Mirror descent step with entropy geometry
    # Keeps S inside the simplex automatically
    # =====================================================

    def mirror_step(self, grad):

        eta = self.eta0 / np.sqrt(self.n + 1)

        # log-sum-exp stabilization
        z = np.log(self.S + 1e-16) - eta * grad
        z -= np.max(z)

        self.S = np.exp(z)
        self.S /= self.S.sum()


    # =====================================================
    # One stochastic iteration
    # =====================================================

    def step(self):

        # ---------------------------------
        # Sample subset according to a
        # U ~ a
        # ---------------------------------

        idx = np.random.choice(self.q, p=self.a)
        subset = self.subsets[idx]

        # Pick-Freeze
        Y, YU = pick_freeze_sample(
            self.sampler,
            self.f,
            subset
        )

        # ---------------------------------
        # Online mean update
        # m_{n+1} = m_n + (Y - m_n)/(n+1)
        # ---------------------------------

        self.m += (Y - self.m) / (self.n + 1)

        # gradient
        grad = self.gradient(Y, YU, idx)

        # mirror descent
        self.mirror_step(grad)

        self.n += 1

        return self.S.copy()


    # =====================================================
    # Run the algorithm
    # =====================================================

    def run(self, N):

        history = np.zeros((N, self.q))

        for i in range(N):
            history[i] = self.step()

        return history

In [10]:

# ============================================================
# Example usage
# ============================================================

def sampler():
    return np.random.rand(5)


def f(X):
    return np.sum(X) + np.prod(X)


p = 5
subsets = generate_subsets(p)
q = len(subsets)

# Example: uniform distribution
a = np.ones(q) / q

algo = PickFreezeMirror(
    f=f,
    sampler=sampler,
    p=p,
    eta0=0.1,
    a=a
)

history = algo.run(5000)

print("Final S:", history[-1])


Final S: [0.03656258 0.02987747 0.03109003 0.03310455 0.03091618 0.03054575
 0.03031088 0.03019938 0.03144293 0.03565945 0.02782589 0.02970758
 0.03095893 0.02821585 0.0295042  0.03249887 0.03314396 0.0323186
 0.0293612  0.03283341 0.02790235 0.02599612 0.0364562  0.03224582
 0.03125745 0.03149629 0.02738455 0.03287407 0.03205652 0.0351454
 0.02933254 0.03177501]


In [14]:
# test fonction d'ishigami
def sampler_ishigami():
    return np.random.uniform(-np.pi, np.pi, size=3)

def f_ishigami(X):
    a = 7 
    b = 0.1 
    return np.sin(X[0]) + a * np.sin(X[1])**2 + b * X[2]**4 * np.sin(X[0])

p = 3
subsets = generate_subsets(p) 
q = len(subsets)

a = np.ones(q) / q

algo_ishigami = PickFreezeMirror( f=f_ishigami, sampler=sampler_ishigami, p=p, eta0=0.1, a=a ) 

history_ishigami = algo_ishigami.run(5000) 

print("Final S (Ishigami):", history_ishigami[-1])

Final S (Ishigami): [8.59520711e-01 5.96132074e-04 3.78056841e-03 8.76245556e-02
 7.57421608e-10 2.68782309e-06 4.60912277e-09 4.84753402e-02]
